In [2]:
import os
from os import listdir, path, makedirs
import numpy as np
import re
import pickle
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage, to_tree
from scipy.spatial.distance import squareform
# import Levenshtein
from Bio import Phylo, SeqIO
from Bio.Align import PairwiseAligner
from Bio.Seq import Seq
from Bio.Phylo.BaseTree import Clade
from Bio.SeqUtils import seq3
from skbio import TreeNode
import matplotlib.pyplot as plt
import argparse
from collections import defaultdict
import pandas as pd
import shutil
import sys
# import time
from datetime import datetime
from multiprocessing import Pool, cpu_count
from itertools import combinations
from scipy.sparse import lil_matrix

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from PIL import Image

from sklearn.metrics.pairwise import euclidean_distances
from sklearn.metrics import silhouette_score
from intervaltree import Interval, IntervalTree
import pandas as pd
import logging

In [3]:
def check_length_condition(value, start_limit, end_limit):
    numbers = list(map(int, value.split('|')))
    return all(num >= start_limit and num <= end_limit for num in numbers)

def has_overlap(value, blockID_set):
    if value == "N":  
        return False
    numbers = set(value.split('|'))
    return not numbers.isdisjoint(blockID_set)

def save_filtered_table(output_folder, start_limit, end_limit, coverage_limit):
    df = pd.read_csv(os.path.join(output_folder,"final_all_blocks_table.csv"), sep=',')
    df_sym = df[df["symmetric"] == "Y"].copy()
    df_not_sym = df[df["symmetric"] == "N"].copy()
    df_filtered_not_sym = df_not_sym[(df_not_sym['length'].astype(int) >= start_limit) & (df_not_sym['length'].astype(int) <= end_limit) & (df_not_sym['coverage'].astype(int) > coverage_limit)]
    df_filtered_sym = df_sym[(df_sym['length'].apply(lambda length_value: check_length_condition(length_value, start_limit, end_limit))) & (df_sym['coverage'].astype(int) > coverage_limit)]
    blockID_set = set(df_filtered_sym["blockID"])
    df_overlap = df_filtered_not_sym[df_filtered_not_sym["within_symmetric"].apply(lambda x: has_overlap(x, blockID_set))]
    df_filtered_not_sym_no_overlap = df_filtered_not_sym[~df_filtered_not_sym["within_symmetric"].apply(lambda x: has_overlap(x, blockID_set))]
    print("final not symmetric count (include overlap):",len(df_filtered_not_sym))
    print("final overlap count:", len(df_overlap))
    print("final symmetric number count (include overlap):",len(df_filtered_sym))
    df_filtered_combined = pd.concat([df_filtered_not_sym_no_overlap, df_filtered_sym], axis=0, ignore_index=True)
    df_filtered_combined = df_filtered_combined.sort_values(by="blockID", ascending=True)
    df_filtered_combined.to_csv(os.path.join(output_folder,f"final_filtered_blocks_table_{start_limit}_{end_limit}_{coverage_limit}.csv"), index=False)  
    

In [4]:
output_folder = "../../../../human_test_samples_output/SRR1853808_90_90_16_48/additional_step"
save_filtered_table(output_folder,16,25,1)

final not symmetric count (include overlap): 2162
final overlap count: 0
final symmetric number count (include overlap): 73


In [6]:
output_folder = "../../../../human_test_samples_output/SRR1853808_90_90_16_48/additional_step"
save_filtered_table(output_folder,16,25,10)

final not symmetric count (include overlap): 369
final overlap count: 0
final symmetric number count (include overlap): 35


In [7]:
output_folder = "../../../../human_test_samples_output/SRR1853808_100_100_16_48/additional_step"
save_filtered_table(output_folder,16,25,1)

final not symmetric count (include overlap): 1307
final overlap count: 0
final symmetric number count (include overlap): 72


In [8]:
output_folder = "../../../../human_test_samples_output/SRR1853808_100_100_16_48/additional_step"
save_filtered_table(output_folder,16,25,10)

final not symmetric count (include overlap): 233
final overlap count: 0
final symmetric number count (include overlap): 38
